### FAISS Vector Store DB:
Faiss (Facebook AI Similarity Search) is an open-source library by Meta for efficient similarity search and clustering of high-dimensional dense vectors, crucial for tasks like recommendation systems, image/text retrieval, and natural language processing. It handles massive datasets (millions to billions of vectors) by using optimized indexing (like Product Quantization) and Approximate Nearest Neighbor (ANN) search, offering both CPU and GPU support for speed, making it far faster than traditional databases for finding similar items. 

In [15]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings.ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

## 1. Load the text document (Data Ingestion)
text_loader = TextLoader('speech.txt')
text_doc = text_loader.load()

## 2. Split the text document into smaller chunks (Data Transformation)
text_splitter = CharacterTextSplitter(
    chunk_size = 10,
    chunk_overlap  = 5,
)
text_chunks = text_splitter.split_documents(text_doc)

In [16]:
## 3. Convert text chunks into embeddings and store them in a vector store (Data Storage)
embeddings = OllamaEmbeddings() 
data_base = FAISS.from_documents(text_chunks, embeddings)

In [13]:
## 4. Query the vector store (Data Querying)
query = "What is the importance of managing our time?"
response = data_base.similarity_search(query)


### Retriever
Convert vector store into retriever class to easily use it with langchain with any LLM model.
retriever is an interface with vector store database

In [9]:
retriever = data_base.as_retriever()
retriever.invoke(query)

[Document(id='4e3ff468-af2c-4a86-8757-9f1c81ca1043', metadata={'source': 'speech.txt'}, page_content='"Good morning everyone.\nTime is our most valuable resource—it is the only thing we can never get back once it is gone.\nWe often waste time waiting for the \'perfect\' moment, but the truth is that the best time to start anything is right now.\nBy managing our time wisely, we can achieve our goals and still have room for the people we love. \nLet’s make every second count. Thank you."')]

In [17]:
response_with_score = data_base.similarity_search_with_score(query)
response_with_score

[(Document(id='24be2b28-ed82-4248-b21d-e7fc858198dc', metadata={'source': 'speech.txt'}, page_content='"Good morning everyone.\nTime is our most valuable resource—it is the only thing we can never get back once it is gone.\nWe often waste time waiting for the \'perfect\' moment, but the truth is that the best time to start anything is right now.\nBy managing our time wisely, we can achieve our goals and still have room for the people we love. \nLet’s make every second count. Thank you."'),
  np.float32(10342.07))]

### embedding vector and do similarity seacrch with vector

In [18]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[-0.13210849463939667,
 -0.19732193648815155,
 -0.3554053008556366,
 0.2134813368320465,
 -2.5061192512512207,
 2.350705623626709,
 0.3557066023349762,
 -0.4281032085418701,
 -1.337058424949646,
 -1.8438117504119873,
 1.197512149810791,
 -1.2776200771331787,
 -0.7524945735931396,
 0.2832728326320648,
 -1.4265974760055542,
 -1.5792714357376099,
 -0.7829551696777344,
 0.9140893220901489,
 1.6289668083190918,
 -2.079280376434326,
 -0.15565907955169678,
 0.22842642664909363,
 0.7295874953269958,
 -1.3166742324829102,
 1.9307109117507935,
 0.10122570395469666,
 -0.039526425302028656,
 -0.7117120027542114,
 -0.43469440937042236,
 -2.0351600646972656,
 2.581407308578491,
 -2.1676619052886963,
 -0.5716628432273865,
 5.330914497375488,
 2.0375850200653076,
 -4.703848361968994,
 0.2793578803539276,
 1.8133974075317383,
 0.9523554444313049,
 -1.1976714134216309,
 -0.9964180588722229,
 -2.437176465988159,
 0.4135269522666931,
 -1.492463231086731,
 -1.2297122478485107,
 1.0710670948028564,
 -2.1909

In [19]:
response_with_score = data_base.similarity_search_by_vector(embedding_vector)
response_with_score

[Document(id='24be2b28-ed82-4248-b21d-e7fc858198dc', metadata={'source': 'speech.txt'}, page_content='"Good morning everyone.\nTime is our most valuable resource—it is the only thing we can never get back once it is gone.\nWe often waste time waiting for the \'perfect\' moment, but the truth is that the best time to start anything is right now.\nBy managing our time wisely, we can achieve our goals and still have room for the people we love. \nLet’s make every second count. Thank you."')]

In [20]:
## Saving and loading the vector store (Data Persistence)
data_base.save_local("faiss_index")

In [22]:
new_data_base = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization = True)
docs = new_data_base.similarity_search(query)

In [23]:
docs

[Document(id='24be2b28-ed82-4248-b21d-e7fc858198dc', metadata={'source': 'speech.txt'}, page_content='"Good morning everyone.\nTime is our most valuable resource—it is the only thing we can never get back once it is gone.\nWe often waste time waiting for the \'perfect\' moment, but the truth is that the best time to start anything is right now.\nBy managing our time wisely, we can achieve our goals and still have room for the people we love. \nLet’s make every second count. Thank you."')]